In [1]:
import networkx as nx
import numpy as np
import functools

In [2]:
@functools.lru_cache(maxsize=None)  # 没有缓存限制
def shortest_path_from_node(graph, source) -> dict['Node' : 'distance']:
    '''
    计算源节点节点source到其他节点的最短路径，包括源节点本身
    返回值为dict[source node to target node, distance]
    '''
    
    return nx.shortest_path_length(graph, source)

@functools.lru_cache(maxsize=None)  # 没有缓存限制
def all_kshell_values(graph: nx.Graph) -> dict['Node' : 'k_shell_value']:
    
    return nx.core_number(graph)

@functools.lru_cache(maxsize=None)  # 没有缓存限制
def nodes_degree(graph) -> dict['Node' : 'degree']:
    
    return dict(graph.degree())

In [3]:
def threshold_lambda_c(G):
    """
    计算SI模型在网络上的传播阈值 lambda_c
    :param graph: 网络图（networkx.Graph）
    :return: 传播阈值 lambda_c
    """
    # 获取网络的度序列
    degrees = [degree for node, degree in G.degree()]

    # 计算 <k> 和 <k^2>
    k_mean = np.mean(degrees)  # 平均度 <k>
    k2_mean = np.mean([k**2 for k in degrees])  # 度的平方的平均值 <k^2>

    # 根据公式计算传播阈值
    lambda_c = k_mean / k2_mean 
    
    return lambda_c

In [4]:
from collections import defaultdict
import networkx as nx

def yita(graph: nx.Graph, i: 'node', degree: dict = None) -> dict[int, dict['node', float]]:
    """
    计算节点 i 的不同层级邻居的归一化度系数 (η)。
    
    :param graph: 网络图 (networkx.Graph)
    :param i: 中心节点
    :param degree: 可选参数，节点的度字典 {node: degree}
    :return: 字典 {layer: {node: η}}，表示每个层级的归一化系数
    """
    # 输入检查
    if not isinstance(graph, nx.Graph):
        raise TypeError("输入必须是一个 networkx.Graph 对象")
    if i not in graph:
        raise ValueError(f"节点 {i} 不在图中")
    if len(graph) == 0:
        raise ValueError("图不能为空")

    # 如果未提供度字典，则计算所有节点的度
    if degree is None:
        degree = dict(graph.degree())

    # 获取节点 i 到其他节点的最短路径
    shortest_paths_from_i = nx.single_source_shortest_path_length(graph, i)

    # 按层级分组
    layers = defaultdict(set)
    for node, distance in shortest_paths_from_i.items():
        if distance > 0:  # 排除自身
            layers[distance].add(node)

    # 计算每一层的归一化系数
    result = {}
    for layer, nodes in layers.items():
        if not nodes:
            continue
        max_degree = max(degree[node] for node in nodes)
        result[layer] = {node: degree[node] / max_degree for node in nodes}

    return result

In [5]:
import networkx as nx

def hierarchy_score_model(graph: nx.Graph, i: 'node') -> float:
    '''
    计算节点 i 的层级传播得分。
    cs(u,x,i) = pro(u,x,i) * [1 + η(u,x,i)] * ks(x)
    u 是源节点，x 是 u 的第 i 层邻居节点。
    '''
    # 计算传播阈值
    belta = threshold_lambda_c(graph)
    
    # 获取节点 i 到其他节点的最短路径
    shortest_paths_from_i = nx.single_source_shortest_path_length(graph, i)

    # 获取每个节点的 k-shell 值
    k_shell = all_kshell_values(graph)

    # 获取节点 i 的邻居分层及 η 值
    yita_values = yita(graph, i)

    # 提取不同层级的邻居节点
    layers = {
        1: set(node for node, distance in shortest_paths_from_i.items() if distance == 1),
        2: set(node for node, distance in shortest_paths_from_i.items() if distance == 2),
        3: set(node for node, distance in shortest_paths_from_i.items() if distance == 3)
    }

    # 如果没有第 1 层邻居，直接返回 0
    if not layers[1]:
        return 0.0

    # 计算每一层的传播得分
    total_score = 0.0

    # 第 1 层传播得分
    propagated_1 = belta
    for node_1 in layers[1]:
        eta_1 = yita_values.get(1, {}).get(node_1, 0.0)  # 获取 η 值，默认为 0.0
        cs_1 = propagated_1 * (1 + eta_1) * k_shell[node_1]
        total_score += cs_1

    # 如果没有第 2 层邻居，返回第 1 层的得分
    if not layers[2]:
        return total_score

    # 计算第 2 层传播概率
    propagated_2 = {}
    for node_2 in layers[2]:
        # 计算该节点与第 1 层邻居的交集数量
        number_path2 = len(layers[1] & set(graph.neighbors(node_2)))
        # 计算未被传播的概率
        unpro_2 = (1 - belta ** 2) ** number_path2
        # 计算被传播的概率
        propagated_2[node_2] = 1 - unpro_2

    # 第 2 层传播得分
    for node_2 in layers[2]:
        eta_2 = yita_values.get(2, {}).get(node_2, 0.0)  # 获取 η 值，默认为 0.0
        cs_2 = propagated_2[node_2] * (1 + eta_2) * k_shell[node_2]
        total_score += cs_2

    # 如果没有第 3 层邻居，返回第 1 层和第 2 层的得分
    if not layers[3]:
        return total_score

    # 计算第 3 层传播概率
    propagated_3 = {}
    for node_3 in layers[3]:
        # 获取与该节点最近的第 2 层邻居节点
        latest_node = layers[2] & set(graph.neighbors(node_3))
        # 计算未被传播的概率
        unpropagated_3 = 1.0
        for node_2 in latest_node:
            unpropagated_current_path = 1 - propagated_2[node_2] * belta
            unpropagated_3 *= unpropagated_current_path
        # 计算被传播的概率
        propagated_3[node_3] = 1 - unpropagated_3

    # 第 3 层传播得分
    for node_3 in layers[3]:
        eta_3 = yita_values.get(3, {}).get(node_3, 0.0)  # 获取 η 值，默认为 0.0
        cs_3 = propagated_3[node_3] * (1 + eta_3) * k_shell[node_3]
        total_score += cs_3

    # 返回所有层级的传播得分之和
    return total_score

In [ ]:
def local_propagation_probability(graph)-> dict['node','score']:
    lpp = {}
    for node in graph.nodes():
        lpp[node] = hierarchy_score_model(graph, node)
    return lpp